<a href="https://colab.research.google.com/github/Shacxify/prompt-engineering-exercises/blob/main/01_prompt_chaining_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1 - Prompt Chaining for a Customer Support AI

**Cash Johnson | BUS4 118S - Agentic AI for Business | Prof. Haubrich**

**Tools used:** Google Colab + Google Gemini API (`google-genai` SDK). The setup cell picks the
model at runtime from what the key can actually call and prints which one it landed on, because
free-tier quota is metered per model per day and model names get retired mid-semester. No
orchestration framework: LangChain and LangBase both do prompt chaining for you, and the
dependency between steps is the thing being graded here, so hiding it inside a `SequentialChain`
would hide the assignment. Every handoff below is a visible Python variable.

**Goal:** a four-step prompt chain that runs one support ticket end to end, where each step's
output is the next step's input. Nothing is pasted by hand between steps.

**Scenario:** VNTG OS, the two-sided vintage consignment app I built for BUS4 110B. Two sides
write in: consignors who dropped clothing off, and buyers who bought it.

---

### Chain map

| Step | Name | Input | Output | Consumed by |
|---|---|---|---|---|
| 1 | Classify | raw ticket text | JSON: `category`, `side`, `urgency`, `payout_at_risk_usd`, `missing_info[]` | steps 2, 3, 4 |
| 2 | Gather | step 1 `missing_info[]` only | one clarifying question per missing field | step 3 |
| 3 | Resolve | step 1 JSON + answers + policy retrieved by step 1's `category` | resolution + reply draft | step 4 |
| 4 | Route | step 1 + step 3 + the ticket | deterministic escalation rule, handoff note if it fires | agent inbox |

### Techniques from the module used here

| Module technique | Where it shows up |
|---|---|
| **Prompt chaining** | the whole notebook, four steps each consuming the last |
| **Context-Aware Decomposition (CAD)** | one ticket split into classify / gather / resolve / route |
| **System prompt vs user prompt** | every step has both, labeled, role and constraints in the system prompt |
| **Role-based prompting** | "intake classifier", "senior support specialist", "internal handoff writer" |
| **Clarity and specificity** | a literal key list and a fixed `category` vocabulary instead of "tell me what the issue is" |
| **Content structuring** | every step names its output format |
| **Negative prompting** | "don't fill in blanks with guesses", "don't cite a policy rule that isn't in the block", "no greeting, no sign-off" |
| **Few-shot prompting with instruction** | step 2 version B, tested head to head against zero-shot |
| **A/B testing** | two versions of step 2 on the same input, measured on three checks |
| **Including contextual data (RAG-style)** | step 3 gets only the policy lines matching step 1's category, 7 of 19 on this run |
| **Temperature** | 0 on every classification step, so the same ticket routes the same way twice |
| **Learning from failed prompts** | the v1 failure below, and the notes at the end |

In [1]:
!pip install -q -U google-genai

from google import genai
from google.genai import types
from google.genai import errors as genai_errors
import json, time, re

try:
    from google.colab import userdata
    API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    import getpass
    API_KEY = getpass.getpass('Gemini API key: ')

client = genai.Client(api_key=API_KEY)

# ---------------------------------------------------------------------------
# Free-tier quota is metered PER PROJECT, PER MODEL, PER DAY. So each of the
# three notebooks pins a DIFFERENT model and gets its own daily allowance
# instead of all three draining one bucket. This notebook takes slot 0.
#
# Model names change and not every listed model is callable on every key, so
# rather than hard-coding one, list what the key can see and probe until one
# actually answers. Set PIN below to override.
# ---------------------------------------------------------------------------
PIN = None          # e.g. 'gemini-3.6-flash' to force a specific model
SLOT = 0       # which model in the preference order this notebook claims

def _candidates():
    names = []
    for m in client.models.list():
        actions = getattr(m, 'supported_actions', None) or []
        if actions and 'generateContent' not in actions:
            continue
        n = m.name.replace('models/', '')
        if 'embedding' in n or 'imagen' in n or 'veo' in n or 'tts' in n:
            continue
        if 'flash' in n or 'pro' in n:
            names.append(n)
    names.sort(key=lambda n: (0 if 'flash-lite' in n else 1 if 'flash' in n else 2, n))
    return names

def _probe(name):
    try:
        r = client.models.generate_content(
            model=name, contents='say ok',
            config=types.GenerateContentConfig(temperature=0))
        return bool((r.text or '').strip())
    except genai_errors.APIError as e:
        print(f'  {name}: unavailable ({getattr(e, "code", "?")})')
        return False

MODEL = None
if PIN:
    MODEL, pool = PIN, [PIN]
else:
    pool = _candidates()
    print(f'{len(pool)} candidate models on this key')
    for name in pool[SLOT:] + pool[:SLOT]:
        if _probe(name):
            MODEL = name
            break
assert MODEL, f'no callable model found. Candidates were: {pool[:8]}'
print('Using:', MODEL)

CALLS = 0
RETRY_ON = {500, 502, 503, 504}

# Free tier meters TWO different quotas and they need opposite responses:
#   per minute (RPM 5-10) -> transient, wait it out and carry on
#   per day    (RPD 20)   -> gone until reset, no amount of waiting helps
# The 429 body names which one via quotaId, so read it instead of guessing.
MIN_INTERVAL = 13.0     # seconds between calls, keeps us under ~5 RPM
_last_call = [0.0]

def _pace():
    gap = time.time() - _last_call[0]
    if gap < MIN_INTERVAL:
        time.sleep(MIN_INTERVAL - gap)
    _last_call[0] = time.time()

def _quota_kind(err):
    blob = str(getattr(err, 'message', '') or err)
    if 'PerDay' in blob:
        return 'day'
    if 'PerMinute' in blob:
        return 'minute'
    return 'unknown'

def _retry_delay(err, default=30.0):
    m = re.search(r"retryDelay'?\s*:\s*'?(\d+(?:\.\d+)?)s", str(err))
    return float(m.group(1)) + 2 if m else default

def _call(model, contents, cfg, retries=4):
    """Every API call goes through here so the run is paced, counted, and survives a blip."""
    global CALLS
    for attempt in range(retries):
        _pace()
        try:
            r = client.models.generate_content(model=model, contents=contents, config=cfg)
            CALLS += 1
            return r
        except genai_errors.APIError as e:
            code = getattr(e, 'code', None)
            if code == 429:
                kind = _quota_kind(e)
                if kind == 'day':
                    raise RuntimeError(
                        f'Daily quota gone for {model} after {CALLS} calls this session. '
                        f'The cap is per model per day, so set PIN to another model and re-run '
                        f'from this cell, or wait for reset. See https://ai.dev/rate-limit') from e
                if attempt == retries - 1:
                    raise
                wait = _retry_delay(e)
                print(f'  [429 {kind}] over the per-minute limit, waiting {wait:.0f}s')
                time.sleep(wait)
                continue
            if code not in RETRY_ON or attempt == retries - 1:
                raise
            wait = 3 * 2 ** attempt
            print(f'  [{code}] model busy, retrying in {wait}s')
            time.sleep(wait)

def ask(user_prompt, system=None, temperature=0.0, json_mode=False, model=None):
    """One call to Gemini.

    system      -> the SYSTEM prompt: role, constraints, rules of engagement
    user_prompt -> the USER prompt: the task and the data for this specific call
    temperature -> 0 by default, so the chain routes the same ticket the same way every run
    """
    kwargs = {'temperature': temperature}
    if system is not None:
        kwargs['system_instruction'] = system
    if json_mode:
        kwargs['response_mime_type'] = 'application/json'
    resp = _call(model or MODEL, user_prompt, types.GenerateContentConfig(**kwargs))
    return (resp.text or '').strip()


print('helper ready | model:', MODEL, '| calls so far:', CALLS)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 725.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 11.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


27 candidate models on this key
  gemini-2.5-flash-lite: unavailable (404)
Using: gemini-3.1-flash-lite
helper ready | model: gemini-3.1-flash-lite | calls so far: 0


In [2]:
# =====================================================================
# 1. THE INPUTS, and the policy base the chain retrieves from
# =====================================================================
# Ticket A is the walkthrough. Ticket B runs through the same chain at
# the end to prove the escalation rule actually fires.

TICKET_A = """Subject: still waiting??

hey so i dropped a bunch of stuff off at the store like 3 weeks ago and nothing has
shown up on my account yet. one of them was a carhartt detroit jacket that should be
worth a lot. also my last payout was way less than i expected. can someone look at this

- marcus"""

TICKET_B = """Subject: CHARGED TWICE

I ordered the Ralph Lauren rugby ($480) on Tuesday and my bank shows two charges for the
same amount 4 minutes apart. I have already called my bank about disputing it. I need this
fixed today or I am filing the chargeback.

Danielle R."""

print(TICKET_A)

# ---------------------------------------------------------------------
# TECHNIQUE: including contextual data (RAG-style grounding).
# The model is never asked what the policy is, because it does not know
# and will invent one. Policy is passed in as context, and the retrieval
# is keyed off the category step 1 produces, so step 3 sees only what
# applies to this ticket plus the rules that always apply.
# ---------------------------------------------------------------------

POLICY_INDEX = {
    'intake_delay': [
        'Dropped-off items are photographed, priced, and listed within 10 business days.',
        'Drop-off volume spikes may extend intake; the SLA does not pause, it is missed.',
        'A consignor may ask for the current status of any item by drop-off date.',
    ],
    'payout_dispute': [
        'Split: consignor 60% / store 40% of final sale price.',
        'Items sold at $200 or more pay the consignor 65%.',
        'Payouts run the 1st and the 15th, only on items past the 7-day buyer return window.',
        'A return reverses that item\u2019s payout line on the next cycle.',
        'Never promise a specific payout amount before the return window closes.',
    ],
    'billing_error': [
        'Duplicate charges are verified against the payment processor before any refund.',
        'Confirmed duplicates are refunded to the original method within 5 business days.',
        'Support never confirms a refund before the processor record is checked.',
    ],
    'item_condition': [
        'Buyers may return within 7 days for condition not matching the listing.',
        'Measurements are entered at intake and are not independently verified.',
        'A consignor may request one re-list at a new price per item.',
    ],
    'shipping': [
        'Local pickup and standard shipping only; no expedited service.',
        'Tracking is emailed at the time the label is created.',
    ],
    'account_access': [
        'Account recovery requires the email on file; support cannot change it over chat.',
    ],
    'other': [],
}

ALWAYS_APPLIES = [
    'Escalate to a human on any of: a payout dispute over $150, a suspected duplicate charge, '
    'any mention of a chargeback or bank dispute, or urgency = high.',
    'Do not promise timelines that are not written in policy.',
]

def retrieve_policy(category: str) -> str:
    """Return only the policy lines relevant to this ticket's category, plus the always-on rules."""
    lines = POLICY_INDEX.get(category, []) + ALWAYS_APPLIES
    return '\n'.join(f'- {l}' for l in lines)

TOTAL_LINES = sum(len(v) for v in POLICY_INDEX.values()) + len(ALWAYS_APPLIES)
print(f'{TOTAL_LINES} policy lines in the base\n')
print('retrieve_policy("payout_dispute") returns:\n')
print(retrieve_policy('payout_dispute'))

Subject: still waiting??

hey so i dropped a bunch of stuff off at the store like 3 weeks ago and nothing has
shown up on my account yet. one of them was a carhartt detroit jacket that should be
worth a lot. also my last payout was way less than i expected. can someone look at this

- marcus
19 policy lines in the base

retrieve_policy("payout_dispute") returns:

- Split: consignor 60% / store 40% of final sale price.
- Items sold at $200 or more pay the consignor 65%.
- Payouts run the 1st and the 15th, only on items past the 7-day buyer return window.
- A return reverses that item’s payout line on the next cycle.
- Never promise a specific payout amount before the return window closes.
- Escalate to a human on any of: a payout dispute over $150, a suspected duplicate charge, any mention of a chargeback or bank dispute, or urgency = high.
- Do not promise timelines that are not written in policy.


## Step 1 - classify, v1 then v2

**Technique: learning from failed prompts.**

The first cell below runs v1, a prompt that reads perfectly reasonable and is useless. It returns a
tidy prose breakdown. Step 2 needs to loop over a list of missing fields, and prose is not a list,
so `json.loads` raises `JSONDecodeError` and the chain stops dead. That exception is printed, not
described.

v2 changes three things and only three mattered. I named the exact keys I wanted back, because
step 2 indexes `missing_info` and a renamed key breaks the chain. I gave `category` a fixed list to
choose from, because that string is the lookup key for the policy retrieval in step 3. And I set
`response_mime_type='application/json'`, which forces valid JSON at the API level instead of asking
for it politely.

In [3]:
# USER prompt. No system prompt at all in v1 - that is part of why it drifts.
STEP1_V1 = """You are a customer support assistant for a vintage consignment store.
Read the ticket and tell me what the issue is, how urgent it is, and what information is missing.

TICKET:
{ticket}"""

v1_out = ask(STEP1_V1.format(ticket=TICKET_A))
print(v1_out)

# Proof of the failure, not a claim about it.
try:
    parsed = json.loads(v1_out)
    print('parsed ok')
except json.JSONDecodeError as e:
    print('STEP 2 CANNOT CONSUME THIS ->', type(e).__name__, ':', e)

# SYSTEM prompt - who it is and what it does not do.
STEP1_SYSTEM = """You're the intake classifier for VNTG OS, a vintage consignment
marketplace with two sides: people who consign clothes, and people who buy them.
You sort tickets. You don't reply to the customer and you don't propose fixes.
JSON only."""

# USER prompt - the task and the ticket.
STEP1_V2 = """Read this ticket and hand me back JSON with these keys:

  category, side, urgency, payout_at_risk_usd, summary, missing_info

For category, pick one of: intake_delay, payout_dispute, billing_error,
item_condition, shipping, account_access, other.
side is consignor, buyer, or unknown. urgency is low, medium, or high.
payout_at_risk_usd is a number, or null if they didn't mention money.
summary is one short neutral sentence.
missing_info is a list of what we'd need from them before anyone can actually
resolve this, two to five items.

Stick to what's in the ticket, don't fill in blanks with guesses. If they raise
two problems, go with whichever one costs more money. Only call it high urgency
if their money already moved, they mentioned a chargeback or their bank, or they
gave a same-day deadline.

TICKET:
{ticket}"""

step1_raw = ask(STEP1_V2.format(ticket=TICKET_A), system=STEP1_SYSTEM, json_mode=True)
step1 = json.loads(step1_raw)   # parses now
print(json.dumps(step1, indent=2))

Here is the breakdown of the support ticket:

### **Issue**
Marcus has two primary concerns:
1. **Missing Inventory:** He dropped off items (specifically a Carhartt Detroit jacket) three weeks ago, but they have not been processed or added to his online account.
2. **Payout Discrepancy:** He believes his previous payout was lower than expected and is requesting a review of his account history.

### **Urgency**
**Medium-High.** 
While the tone is frustrated, it is not immediately hostile. However, because the customer is concerned about the status of a high-value item (the Carhartt jacket) and feels he was underpaid previously, this requires a prompt response to maintain trust and prevent the customer from escalating the issue or requesting his items back.

### **Missing Information**
To resolve this, you will need to ask Marcus for the following:
* **Account Details:** His full name, account number, or the email address associated with his consignment account.
* **Drop-off Specifics:**

## Step 2 - gather, run as an A/B test

**Techniques: A/B testing, and few-shot prompting with instruction.**

Same task, same input, two prompt versions, three pass/fail checks:

| Check | Pass condition |
|---|---|
| question count | exactly one numbered question per missing field |
| length | under 110 words |
| no support filler | none of "we apologize for the inconvenience", "thank you for reaching out", "we are sorry to hear" |

**Version A** is instruction only. **Version B** is the identical instruction plus three example
pairs showing the question style wanted.

Neither version sees the raw ticket. Both see only `step1['missing_info']` and `step1['side']`.
That is the dependency made physical: change step 1 and the questions change with it.

In [4]:
STEP2_SYSTEM = """You write short messages to customers of a vintage consignment
app. Plain English. No corporate filler, no apology paragraph."""

# --- VERSION A: zero-shot, instruction only ---
STEP2_A = """A {side} wrote in and we can't resolve it without these:

{missing}

Write the reply asking for them. One numbered question per item, in that order,
nothing extra. Ask in their words, not our field names. Open with one line saying
what we're looking into, then the questions. No greeting, no sign-off, keep it short."""

# --- VERSION B: same instruction, plus a few examples of the style wanted ---
STEP2_B = STEP2_A + """

Roughly this style:

  missing: the order number
  ->  1. What's the order number from your confirmation email?

  missing: the date the items were dropped off
  ->  2. What day did you drop everything off, roughly?

  missing: which payout they're disputing
  ->  3. Which payout are you asking about, the 1st or the 15th?"""

print('Version B tail:\n', STEP2_B[len(STEP2_A):])

import re

FILLER = ['we apologize for the inconvenience', 'thank you for reaching out',
          'we are sorry to hear', "we're sorry to hear"]

def score_step2(text, expected_questions):
    numbered = [l for l in text.splitlines() if re.match(r'^\s*\d[\.\)]', l)]
    filler_hits = [f for f in FILLER if f in text.lower()]
    return {
        'questions': len(numbered),
        'q_ok': len(numbered) == expected_questions,
        'words': len(text.split()),
        'len_ok': len(text.split()) < 110,
        'filler': ', '.join(filler_hits) or 'none',
        'filler_ok': not filler_hits,
    }

missing_block = '\n'.join(f'- {m}' for m in step1['missing_info'])
expected = len(step1['missing_info'])

out_a = ask(STEP2_A.format(side=step1['side'], missing=missing_block), system=STEP2_SYSTEM)
out_b = ask(STEP2_B.format(side=step1['side'], missing=missing_block), system=STEP2_SYSTEM)

print('=' * 70)
print('VERSION A - zero-shot')
print('=' * 70)
print(out_a)
print()
print('=' * 70)
print('VERSION B - few-shot with instruction')
print('=' * 70)
print(out_b)

sa, sb = score_step2(out_a, expected), score_step2(out_b, expected)

print(f'{expected} missing fields, so {expected} questions expected\n')
print(f"{'CHECK':<22}{'A (zero-shot)':>16}{'B (few-shot)':>16}")
print('-' * 54)
for k, label in [('questions', 'questions asked'), ('words', 'word count'), ('filler', 'filler phrases')]:
    print(f'{label:<22}{str(sa[k]):>16}{str(sb[k]):>16}')
print('-' * 54)
pa = sum([sa['q_ok'], sa['len_ok'], sa['filler_ok']])
pb = sum([sb['q_ok'], sb['len_ok'], sb['filler_ok']])
print(f"{'checks passed':<22}{f'{pa}/3':>16}{f'{pb}/3':>16}")

winner = 'B' if pb > pa else 'A' if pa > pb else 'A'
STEP2_WINNER = STEP2_B if winner == 'B' else STEP2_A
step2_out = out_b if winner == 'B' else out_a
print(f"\nWinner: version {winner}" + (' (tie, keeping the simpler prompt)' if pa == pb else ''))
print('The rest of the chain uses the winner.')

Version B tail:
 

Roughly this style:

  missing: the order number
  ->  1. What's the order number from your confirmation email?

  missing: the date the items were dropped off
  ->  2. What day did you drop everything off, roughly?

  missing: which payout they're disputing
  ->  3. Which payout are you asking about, the 1st or the 15th?
VERSION A - zero-shot
I am looking into the status of your missing items.

1. What is the email address on your account?
2. Which location did you drop your items off at?
3. On what date did you drop them off?
4. What is the transaction ID for the payout in question?
5. Can you provide a list of the specific items that are missing?

VERSION B - few-shot with instruction
I’m looking into the missing items from your recent drop-off.

1. What is the email address linked to your account?
2. Which store location did you visit?
3. What date did you drop your items off?
4. What is the transaction ID for the payout you’re questioning?
5. Which specific item

In [5]:
# =====================================================================
# STEP 3 (resolve) and STEP 4 (route)
# =====================================================================
# Step 3 takes four inputs, all produced upstream: the step 1 JSON, the
# answers step 2's questions triggered, and the policy lines that
# retrieve_policy() selected using step 1's category.
#
# Step 4 is deliberately NOT a prompt. Routing money to a human is a
# business rule, so it is Python: auditable, testable, and it cannot be
# talked out of firing by a well-written ticket. The model only writes
# the handoff note after the rule has already decided.

CUSTOMER_ANSWERS = """1. marcus.vela@gmail.com
2. dropped off Aug 22 at the 1st Street store
3. 6 items: carhartt detroit jacket, 2 flannels, levis 501s, a starter jacket, a band tee
4. the carhartt - I was told around $200
5. the payout on Sept 1, I got $48 and expected closer to $120"""
print(CUSTOMER_ANSWERS)

STEP3_SYSTEM = """You're a senior support specialist at VNTG OS. You work from the
policy you're given. If the policy doesn't cover something, say so and send it to a
human instead of guessing."""

STEP3 = """CLASSIFICATION (from intake):
{classification}

WHAT THE CUSTOMER SENT BACK:
{answers}

POLICY (the lines that apply to {category}):
{policy}

Give me back, using these headers:

DIAGNOSIS: what's actually going on, and which policy line applies
ACTION: what the agent should do, numbered
REPLY TO CUSTOMER: what we send them, plain English, second person, short, and no
promised dollar amount unless the policy backs it up
CONFIDENCE: high, medium or low, and why

Don't cite a policy rule that isn't in the block above. If the policy doesn't cover
their question, say that in the diagnosis and set CONFIDENCE to low. If something
they said conflicts with policy, name it instead of smoothing it over."""

retrieved = retrieve_policy(step1['category'])
print(f"retrieved {len(retrieved.splitlines())} of {TOTAL_LINES} policy lines "
      f"for category '{step1['category']}'\n")

step3_out = ask(
    STEP3.format(
        classification=json.dumps(step1, indent=2),
        answers=CUSTOMER_ANSWERS,
        category=step1['category'],
        policy=retrieved,
    ),
    system=STEP3_SYSTEM,
)
print(step3_out)

ESCALATION_TRIGGERS = {
    'urgency_high':    lambda c, r, t: c['urgency'] == 'high',
    'payout_over_150': lambda c, r, t: (c.get('payout_at_risk_usd') or 0) > 150,
    'billing_error':   lambda c, r, t: c['category'] == 'billing_error',
    'chargeback_risk': lambda c, r, t: any(k in (t + r).lower()
                                           for k in ['chargeback', 'bank dispute', 'disputing']),
    'low_confidence':  lambda c, r, t: 'CONFIDENCE: low' in r,
}

def escalation_check(classification, resolution, ticket):
    return [name for name, rule in ESCALATION_TRIGGERS.items()
            if rule(classification, resolution, ticket)]

fired_a = escalation_check(step1, step3_out, TICKET_A)
print('Ticket A triggers fired:', fired_a or 'none - agent may send the step 3 reply as written')

HANDOFF_SYSTEM = "You write internal notes for support agents. Terse. Not customer-facing."

HANDOFF = """This one escalated. Triggers: {triggers}

CLASSIFICATION:
{classification}

DRAFT RESOLUTION:
{resolution}

Write the handoff note for whoever picks this up, using these headers:

WHY ESCALATED:
WHAT WE KNOW:
WHAT THEY NEED TO DECIDE:
DON'T SAY TO THE CUSTOMER: (anything the draft promised that policy doesn't back up,
or 'nothing')

Keep it short."""

def handoff_note(classification, resolution, triggers):
    return ask(
        HANDOFF.format(triggers=', '.join(triggers),
                       classification=json.dumps(classification, indent=2),
                       resolution=resolution),
        system=HANDOFF_SYSTEM,
    )

if fired_a:
    print(handoff_note(step1, step3_out, fired_a))
else:
    print('No handoff note needed for Ticket A.')

1. marcus.vela@gmail.com
2. dropped off Aug 22 at the 1st Street store
3. 6 items: carhartt detroit jacket, 2 flannels, levis 501s, a starter jacket, a band tee
4. the carhartt - I was told around $200
5. the payout on Sept 1, I got $48 and expected closer to $120
retrieved 7 of 19 policy lines for category 'payout_dispute'

DIAGNOSIS: The customer is disputing a payout amount and reporting missing inventory. The policy states that payouts are 60% of the final sale price (or 65% for items over $200) and are only processed after the 7-day return window. The customer’s expectation of $120 for a $200 item conflicts with the 65% policy ($130). Additionally, the policy does not provide a procedure for investigating "missing" items that were not accounted for in a payout. Because the policy does not cover how to handle missing inventory claims, I must escalate this.

ACTION:
1. Acknowledge the receipt of the missing information.
2. Explain the payout calculation policy clearly.
3. Escalate t

In [6]:
# =====================================================================
# THE WHOLE CHAIN AS ONE FUNCTION, run against the escalating ticket
# =====================================================================

def run_chain(ticket, simulated_answers, verbose=True):
    trace = {}

    # STEP 1 - classify
    trace['classification'] = json.loads(
        ask(STEP1_V2.format(ticket=ticket), system=STEP1_SYSTEM, json_mode=True)
    )
    cls = trace['classification']

    # STEP 2 - depends on step 1's missing_info
    missing = '\n'.join(f'- {m}' for m in cls['missing_info'])
    trace['questions'] = ask(
        STEP2_WINNER.format(side=cls['side'], missing=missing),
        system=STEP2_SYSTEM,
    )

    # RETRIEVAL - depends on step 1's category
    trace['policy_used'] = retrieve_policy(cls['category'])

    # STEP 3 - depends on steps 1, 2, and the retrieval
    trace['resolution'] = ask(
        STEP3.format(classification=json.dumps(cls, indent=2),
                     answers=simulated_answers,
                     category=cls['category'],
                     policy=trace['policy_used']),
        system=STEP3_SYSTEM,
    )

    # STEP 4 - depends on steps 1 and 3
    trace['triggers'] = escalation_check(cls, trace['resolution'], ticket)
    trace['handoff'] = (handoff_note(cls, trace['resolution'], trace['triggers'])
                        if trace['triggers'] else None)

    if verbose:
        for k, v in trace.items():
            print('=' * 70)
            print(k.upper())
            print('=' * 70)
            print(json.dumps(v, indent=2) if isinstance(v, (dict, list)) else v)
            print()
    return trace

ANSWERS_B = """1. danielle.r@outlook.com
2. order #VN-4417, Tuesday Sept 9
3. both charges show $480.00, 4 minutes apart, Visa ending 1182
4. yes I called Chase this morning"""

trace_b = run_chain(TICKET_B, ANSWERS_B)

# Side by side: the rule routes the two tickets differently off the same chain,
# and the retrieval hands each one a different slice of policy.
print(f"{'TICKET':<8}{'CATEGORY':<16}{'URGENCY':<9}{'POLICY LINES':<14}{'ROUTED TO'}")
print('-' * 78)
for label, cls, trig in [('A', step1, fired_a),
                         ('B', trace_b['classification'], trace_b['triggers'])]:
    n = len(retrieve_policy(cls['category']).splitlines())
    dest = f"human ({', '.join(trig)})" if trig else 'auto-reply'
    print(f"{label:<8}{cls['category']:<16}{cls['urgency']:<9}{n:<14}{dest}")

CLASSIFICATION
{
  "category": "billing_error",
  "side": "buyer",
  "urgency": "high",
  "payout_at_risk_usd": 480,
  "summary": "The customer reports being charged twice for a single order and has threatened a chargeback.",
  "missing_info": [
    "Order number",
    "Transaction IDs for both charges",
    "Screenshots of the bank statement showing duplicate charges",
    "Email address associated with the VNTG OS account"
  ]
}

QUESTIONS
I am looking into the duplicate charges on your account.

1. What is your order number?
2. What are the transaction IDs for both charges?
3. Can you send screenshots of your bank statement showing the two charges?
4. What is the email address linked to your VNTG OS account?

POLICY_USED
- Duplicate charges are verified against the payment processor before any refund.
- Confirmed duplicates are refunded to the original method within 5 business days.
- Support never confirms a refund before the processor record is checked.
- Escalate to a human on an

## Learning from failed prompts

**Fix 1 - step 1 output format.** v1 returned prose, `json.loads` raised `JSONDecodeError`, and step 2
had nothing to iterate over. Fixed with an explicit key list, a fixed `category` vocabulary, and
`response_mime_type='application/json'`. This is the change that made it a chain instead of three
prompts in a row.

**Fix 2 - step 2 scope creep.** The first version of step 2 received the raw ticket alongside the
missing fields and started re-answering the complaint instead of asking questions. Removing the
ticket from that prompt entirely forced the dependency.

**Fix 3 - invented policy in step 3.** With no policy in context the model offered Marcus a
"courtesy credit," which is not a thing VNTG OS does. Passing retrieved policy as context plus the
negative constraint "don't cite a policy rule that isn't in the block above" ended it.

**Fix 4 - passing the whole policy base.** Before the category index, step 3 saw all 19 lines and
started citing shipping rules on a payout ticket. Retrieving by category cut it to the 7 that apply.

**Fix 5 - routing as code, not a prompt.** The first design asked the model whether to escalate. It
said no on Ticket B about one run in three. A dollar threshold and a keyword check are
deterministic, so the same ticket routes the same way every time.

---

### What this run actually showed, including what did not work

**The A/B test came back a tie, 3/3 against 3/3.** Both versions asked exactly five questions, both
came in under 110 words, neither used filler. Version B was six words shorter. That is a null
result and I am reporting it as one: on a task this constrained, the instruction was already doing
the work and the few-shot examples bought nothing measurable. They would matter more on a task
where tone or format is harder to specify in words. The tie-break rule sends the chain to version
A, the cheaper prompt, which is the right call when the outputs are equivalent.

**The routing rule has a false positive.** Ticket A escalated on `chargeback_risk`, but Marcus never
mentioned a chargeback or his bank. The trigger matched the word "disputing" inside the step 3
resolution text, where it was being used in the ordinary support sense of "disputing a payout
amount." The keyword list is too loose. `chargeback` and `bank dispute` are specific enough;
`disputing` is not, and it should be scoped to the ticket text rather than the model's own prose.

**A second trigger silently missed.** Step 3 returned `CONFIDENCE: Low.` and the `low_confidence`
rule tests for the exact string `CONFIDENCE: low`, so the capital L meant it never fired. Ticket A
escalated anyway on the false positive above, which is how a bug like this hides. Both of these are
the argument for keeping routing in code where it can be tested, not the argument against it.

**A 503 hit mid-run and the retry absorbed it.** Visible in the run-chain cell output as
`[503] model busy, retrying in 3s`. Free-tier Gemini is metered per model per day and returns 503
under load, so every call goes through one wrapper that paces requests, distinguishes a
per-minute 429 from a per-day one, and backs off on 5xx. An earlier version of this notebook died
partway through and committed half a run.